In [ ]:
"""
Join Career Launch 2027 Outcomes Placement History to the opportunity
descriptions (JB sheet extract), so every placement row carries the role
description needed for NIOCCS NAICS/SOC classification.

Anchored on the placement history (left join) so we never lose a placement
row, even if a handful of opportunity_ids somehow fail to match.

NOTE ON ENCODING: the opportunities file is Windows-1252 (cp1252), not
UTF-8 -- reading it as UTF-8 will throw a UnicodeDecodeError. This is
common for CSVs exported from Excel/Airtable on Windows.
"""

import pandas as pd

# ---- 1. Load -----------------------------------------------------------
hist = pd.read_csv("Career Launch 2027 Outcomes(Placement Histroy).csv")
opps = pd.read_csv("Career Launch 2027 Outcomes(JB sheet).csv", encoding="cp1252")

print(f"hist rows: {len(hist)}")
print(f"opps rows (before cleanup): {len(opps)}")

# The opps file has a run of fully-empty trailing "Unnamed: N" columns
# from stray commas in the source export -- drop them.
opps = opps.dropna(axis=1, how="all")
print(f"opps columns after dropping empty trailing columns: {list(opps.columns)}")

# The role-title column lost its header in the source export and reads in
# as "Unnamed: 4" -- rename it now so it's usable downstream (e.g. as
# role_name_col for NIOCCS classification).
opps = opps.rename(columns={"Unnamed: 4": "Opportunity Name"})

# ---- 2. Normalize join key ----------------------------------------------
hist["Opportunity Id"] = hist["Opportunity Id"].astype(str).str.strip()
opps["Opportunity Id"] = opps["Opportunity Id"].astype(str).str.strip()

dupe_ids = opps["Opportunity Id"][opps["Opportunity Id"].duplicated(keep=False)]
if not dupe_ids.empty:
    print(f"\n⚠️  WARNING: {dupe_ids.nunique()} opportunity_id(s) appear more than "
          f"once in opps. Investigate before trusting the join.")
else:
    print("\nOpportunity Id is unique in opps -- safe to join on directly.")

# ---- 3. Merge, anchored on hist (left join) ------------------------------
merged = hist.merge(
    opps,
    on="Opportunity Id",
    how="left",
    suffixes=("_hist", "_opps"),
    indicator=True,
)
assert len(merged) == len(hist), f"Row count changed on merge: {len(hist)} -> {len(merged)}"

matched = (merged["_merge"] == "both").sum()
unmatched = (merged["_merge"] == "left_only").sum()
print(f"\nhist rows: {len(hist)} | matched to opps (has description): {matched} | "
      f"unmatched (no description available): {unmatched}")
if unmatched:
    print("Unmatched placements (no Opportunity Description available for these):")
    print(merged.loc[merged["_merge"] == "left_only", ["Opportunity Id", "Agency Name_hist"]].to_string(index=False))

merged = merged.drop(columns="_merge")

# Prefer the opps file's Agency Name (canonical) where available, fall back
# to hist's own Agency Name.
if "Agency Name_opps" in merged.columns:
    merged["Agency Name"] = merged["Agency Name_opps"].fillna(merged["Agency Name_hist"])
else:
    merged["Agency Name"] = merged["Agency Name_hist"]

# ---- 4. Save --------------------------------------------------------------
merged.to_csv("career_launch_2027_with_descriptions.csv", index=False)
print(f"\nSaved {len(merged)} rows -> career_launch_2027_with_descriptions.csv")
print("Ready for classify_dataframe(): use 'Opportunity Name' as role_name_col, "
      "'Opportunity Description' as role_description_col.")

In [ ]:
"""
NIOCCS NAICS/SOC classification for Career Launch 2027 placements.

Input: career_launch_2027_with_descriptions.csv (from join_cl2027_to_opps.py)
Requires: tools/nioccs_classify.py on the path, with NIOCCS_BASE_URL already
fixed to the real CDC endpoint (not the Proofpoint-wrapped URL).

Dedup strategy: unlike BMCC (where hub varied per student for a shared
opportunity), here 'Opportunity Id' is already unique per posting in the
source file, so every hist row sharing an Opportunity Id has IDENTICAL
role name/description/campaign -- dedup on Opportunity Id alone is safe
and collapses 2,992 placement rows down to ~723 unique postings to classify.
"""

import time
import pandas as pd
from tools.nioccs_classify import classify_dataframe

ROLE_NAME_COL = "Opportunity Name_opps"
ROLE_DESC_COL = "Opportunity Description"
CAMPAIGN_COL = "Opportunity Campaign Name"   # e.g. "2026 Career Launch - Healthcare"
INDUSTRY_COL = "Hub"                          # derived: just "Healthcare", not the year/program prefix
ID_COL = "Opportunity Application Id"         # unique per placement/application row

MAX_RETRY_PASSES = 3
RETRY_PAUSE_SECONDS = 1          # was 20
RETRY_SECONDS_BETWEEN_REQUESTS = 0.1   # was 2.0
CLASSIFY_SECONDS_BETWEEN_REQUESTS = 0.05  # was the classify_dataframe default of 0.5
CHECKPOINT_PATH = "career_launch_2027_classify_checkpoint.csv"
CHECKPOINT_EVERY = 50

# ---- 1. Load --------------------------------------------------------------
df = pd.read_csv("career_launch_2027_with_descriptions.csv")
print(f"Loaded {len(df)} rows")

for col in (ROLE_NAME_COL, ROLE_DESC_COL, CAMPAIGN_COL, ID_COL):
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found. Actual columns: {list(df.columns)}")

# Derive Hub from the campaign name: strip a leading "<year/program prefix> - "
# (or ": ") so NIOCCS gets just "Healthcare" / "STEM and Green" / etc. as
# context, not "2026 Career Launch - Healthcare". Falls back to the raw
# campaign string if the pattern doesn't match, and flags anything unparsed
# so a format we haven't seen yet doesn't silently slip through as garbage.
def extract_hub(campaign):
    if pd.isna(campaign):
        return campaign
    text = str(campaign).strip()
    # split on the last " - " or ": " separator, take the tail as the hub
    for sep in (" - ", ": "):
        if sep in text:
            return text.rsplit(sep, 1)[-1].strip()
    return text  # unparsed -- falls back to raw string

df["Hub"] = df[CAMPAIGN_COL].apply(extract_hub)

raw_to_hub = df[[CAMPAIGN_COL, "Hub"]].drop_duplicates().sort_values(CAMPAIGN_COL)
print("\nCampaign -> Hub mapping used (verify this looks right):")
print(raw_to_hub.to_string(index=False))

for col in (ROLE_NAME_COL, ROLE_DESC_COL, INDUSTRY_COL, ID_COL):
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found. Actual columns: {list(df.columns)}")

# ---- 2. Dedupe to unique Opportunity Id (already unique per posting) ------
unique_opps = df.drop_duplicates(subset="Opportunity Id")[
    ["Opportunity Id", ROLE_NAME_COL, ROLE_DESC_COL, INDUSTRY_COL]
].copy()
print(f"Unique opportunities to classify: {len(unique_opps)} "
      f"(saved {len(df) - len(unique_opps)} redundant API calls)")

# ---- 3. Classify (with incremental checkpointing) --------------------------
import os

already_done = pd.DataFrame()
remaining = unique_opps
if os.path.exists(CHECKPOINT_PATH):
    already_done = pd.read_csv(CHECKPOINT_PATH)
    done_ids = set(already_done["Opportunity Id"])
    remaining = unique_opps[~unique_opps["Opportunity Id"].isin(done_ids)]
    print(f"\nResuming from checkpoint: {len(done_ids)} already classified, "
          f"{len(remaining)} remaining.")

buffer = []

def checkpoint(i, total, result_row):
    buffer.append(result_row)
    if i % CHECKPOINT_EVERY == 0 or i == total:
        combined = pd.concat([already_done, pd.DataFrame(buffer)], ignore_index=True)
        combined.to_csv(CHECKPOINT_PATH, index=False)
        print(f"  checkpoint saved: {len(combined)} opportunities classified so far")

if len(remaining) > 0:
    new_results = classify_dataframe(
        remaining,
        role_name_col=ROLE_NAME_COL,
        role_description_col=ROLE_DESC_COL,
        industry_col=INDUSTRY_COL,
        id_col="Opportunity Id",
        seconds_between_requests=CLASSIFY_SECONDS_BETWEEN_REQUESTS,
        on_row_done=checkpoint,
    )
    classified = pd.concat([already_done, new_results], ignore_index=True).set_index("Opportunity Id")
else:
    classified = already_done.set_index("Opportunity Id")

# ---- 4. Retry failures (row-swap merge, not column-wise) ------------------
for attempt in range(1, MAX_RETRY_PASSES + 1):
    failed_mask = classified["API Call Error"].notna() & (classified["Flagged As Blank Intake Form"] != True)
    failed_ids = classified.index[failed_mask]
    if len(failed_ids) == 0:
        break
    print(f"\nRetry pass {attempt}: re-classifying {len(failed_ids)} failed opportunity(ies)...")
    time.sleep(RETRY_PAUSE_SECONDS)

    retry_input = unique_opps[unique_opps["Opportunity Id"].isin(failed_ids)]
    retry_result = classify_dataframe(
        retry_input,
        role_name_col=ROLE_NAME_COL,
        role_description_col=ROLE_DESC_COL,
        industry_col=INDUSTRY_COL,
        id_col="Opportunity Id",
        seconds_between_requests=RETRY_SECONDS_BETWEEN_REQUESTS,
    ).set_index("Opportunity Id")

    classified = classified.drop(index=retry_result.index)
    classified = pd.concat([classified, retry_result])

still_failed = classified["API Call Error"].notna() & (classified["Flagged As Blank Intake Form"] != True)
print(f"\nAfter retries: {still_failed.sum()} opportunity(ies) still failing "
      f"(out of {len(classified)}).")
if still_failed.sum() > 0:
    print("Error message(s):")
    print(classified.loc[still_failed, "API Call Error"].value_counts().to_string())

classified = classified.reset_index()

# ---- 5. Map back onto all placement rows, save -----------------------------
df = df.merge(classified, on="Opportunity Id", how="left")
assert len(df) == 2992, f"Row count changed during merge: expected 2992, got {len(df)}"

df.to_csv("career_launch_2027_classified.csv", index=False)
print(f"\nSaved {len(df)} classified rows -> career_launch_2027_classified.csv")

n_blank = df["Flagged As Blank Intake Form"].sum()
n_error = df["API Call Error"].notna().sum() - n_blank
n_soc_insuff = df["SOC Flagged As Insufficient Information"].fillna(False).sum()
n_naics_insuff = df["NAICS Flagged As Insufficient Information"].fillna(False).sum()
print(f"\nQuality flags (out of {len(df)} placement rows):")
print(f"  blank intake form (skipped):  {n_blank}")
print(f"  API errors:                   {n_error}")
print(f"  SOC insufficient info:        {n_soc_insuff}")
print(f"  NAICS insufficient info:      {n_naics_insuff}")

print("\n--- Quick preview: role type distribution (SOC Title) ---")
valid = df[df["SOC Flagged As Insufficient Information"] != True]
print(valid["SOC Title"].value_counts().head(15).to_string())